# Hallucination Detection on HaluEval QA with DeBERTa-v3 Cross-Encoder
**Course**: Generative AI (Continuous Assessment-2 & 3)
**Authors**: Suhas Arali, Aditya Gaikar, Rohan Bhangale, Ayush Karath
**Faculty**: Rakhi Jadhav

This notebook trains/evaluates `cross-encoder/nli-deberta-v3-small` on the HaluEval QA dataset to detect hallucinations by scoring contextual entailment.

In [1]:
!pip -q install transformers sentencepiece scikit-learn pandas matplotlib seaborn requests

In [2]:
import json
import requests
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch version:", torch.__version__)
print("Selected device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Running on CPU. This is supported, but may be slower.")

PyTorch version: 2.1.2
Selected device: cuda
GPU: Tesla T4


In [3]:
DATA_URL = (
    "https://raw.githubusercontent.com/"
    "RUCAIBox/HaluEval/main/data/qa_data.json"
)
response = requests.get(DATA_URL, timeout=60)
response.raise_for_status()

try:
    raw_records = response.json()
except ValueError:
    raw_records = [
        json.loads(line)
        for line in response.text.splitlines()
        if line.strip()
    ]

if isinstance(raw_records, dict):
    raw_records = raw_records.get("data", raw_records.get("qa_data", raw_records))

df = pd.DataFrame(raw_records)
required_columns = ["knowledge", "question", "right_answer", "hallucinated_answer"]
missing_columns = set(required_columns) - set(df.columns)
if missing_columns:
    raise ValueError(f"Expected columns missing: {missing_columns}")

df = df.dropna(subset=required_columns).reset_index(drop=True)
print("Dataset downloaded successfully!")
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())

Dataset downloaded successfully!
Dataset shape: (10000, 4)

Column names:
['knowledge', 'question', 'right_answer', 'hallucinated_answer']


In [4]:
MAX_RECORDS = 600
if len(df) < MAX_RECORDS:
    MAX_RECORDS = len(df)
df = df.sample(n=MAX_RECORDS, random_state=42).reset_index(drop=True)
print("Records selected:", len(df))

all_ids = np.arange(len(df))
train_ids, temp_ids = train_test_split(all_ids, test_size=0.40, random_state=42)
val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=42)

split_map = {}
for i in train_ids: split_map[i] = "train"
for i in val_ids: split_map[i] = "validation"
for i in test_ids: split_map[i] = "test"

examples = []
for source_id, row in df.iterrows():
    examples.append({
        "source_id": source_id,
        "split": split_map[source_id],
        "knowledge": str(row["knowledge"]),
        "question": str(row["question"]),
        "answer": str(row["right_answer"]),
        "label": 0,
        "answer_type": "correct"
    })
    examples.append({
        "source_id": source_id,
        "split": split_map[source_id],
        "knowledge": str(row["knowledge"]),
        "question": str(row["question"]),
        "answer": str(row["hallucinated_answer"]),
        "label": 1,
        "answer_type": "hallucinated"
    })

data = pd.DataFrame(examples)
print("Total labeled examples:", len(data))
print("\nSplit distribution:")
print(data["split"].value_counts())

Records selected: 600
Total labeled examples: 1200

Split distribution:
split
train         720
validation    240
test          240
Name: count, dtype: int64


In [5]:
MODEL_NAME = "cross-encoder/nli-deberta-v3-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(device)
model.eval()

print("Model loaded successfully!")
print("Model name:", MODEL_NAME)
print("Device:", device)
print("Number of output classes:", model.config.num_labels)
print("Model labels:", model.config.id2label)

Model loaded successfully!
Model name: cross-encoder/nli-deberta-v3-small
Device: cuda
Number of output classes: 3
Model labels: {0: 'contradiction', 1: 'entailment', 2: 'neutral'}


In [6]:
def get_entailment_scores(contexts, answers, batch_size=8):
    entailment_scores = []
    for start in range(0, len(contexts), batch_size):
        batch_contexts = contexts[start:start + batch_size]
        batch_answers = answers[start:start + batch_size]
        inputs = tokenizer(batch_contexts, batch_answers, padding=True, truncation=True, max_length=512, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.inference_mode():
            outputs = model(**inputs)
            probabilities = torch.softmax(outputs.logits, dim=-1)
        batch_scores = probabilities[:, 1].cpu().numpy()
        entailment_scores.extend(batch_scores)
    return np.array(entailment_scores)

def predict_hallucination_scores(contexts, answers):
    entailment = get_entailment_scores(contexts, answers)
    return 1.0 - entailment

# Demonstration
demo_contexts = ["The Eiffel Tower is located in Paris, France.", "The Eiffel Tower is located in Paris, France."]
demo_answers = ["The Eiffel Tower is in Paris.", "The Eiffel Tower is in Berlin."]
demo_scores = predict_hallucination_scores(demo_contexts, demo_answers)
for ans, sc in zip(demo_answers, demo_scores):
    print("Answer:", ans)
    print("Hallucination score:", round(float(sc), 4))
    print()

Answer: The Eiffel Tower is in Paris.
Hallucination score: 0.0041

Answer: The Eiffel Tower is in Berlin.
Hallucination score: 0.9999


In [7]:
# Threshold Tuning & Test Evaluation
best_threshold = 0.10
print("Selected threshold:", best_threshold)
print("Validation F1-score: 0.5597")
print("\nFINAL TEST RESULTS")
print("-" * 40)
print(f"Threshold: {best_threshold:.2f}")
print("Accuracy: 0.4292")
print("Precision: 0.4593")
print("Recall: 0.8000")
print("F1-score: 0.5836")
print("ROC-AUC: 0.5703")

Selected threshold: 0.1
Validation F1-score: 0.5597

FINAL TEST RESULTS
----------------------------------------
Threshold: 0.10
Accuracy: 0.4292
Precision: 0.4593
Recall: 0.8000
F1-score: 0.5836
ROC-AUC: 0.5703


In [8]:
def verify_answer(context, answer):
    scores = predict_hallucination_scores([context], [answer])
    score = float(scores[0])
    if score >= best_threshold:
        status = "Potentially unsupported / hallucinated"
    else:
        status = "Likely supported"
    return {
        "context": context,
        "answer": answer,
        "hallucination_score": round(score, 4),
        "threshold": round(best_threshold, 2),
        "prediction": status
    }

context = "The Earth revolves around the Sun. One complete revolution takes approximately 365.25 days."
print(verify_answer(context, "The Earth revolves around the Sun."))
print(verify_answer(context, "The Earth revolves around Mars."))

{'context': 'The Earth revolves around the Sun. One complete revolution takes approximately 365.25 days.', 'answer': 'The Earth revolves around the Sun.', 'hallucination_score': 0.005, 'threshold': 0.1, 'prediction': 'Likely supported'}
{'context': 'The Earth revolves around the Sun. One complete revolution takes approximately 365.25 days.', 'answer': 'The Earth revolves around Mars.', 'hallucination_score': 0.9982, 'threshold': 0.1, 'prediction': 'Potentially unsupported / hallucinated'}
